In [ ]:
import json
from pathlib import Path

import torch
from huggingface_hub import hf_hub_download
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet50

In [ ]:
REPO_ID = "ibrahimdaud/raw-food-recognition-models"
CLASS_NAMES_PATH = Path("../class_names.json")
DATA_FOLDER = Path("../data")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
checkpoint_path = hf_hub_download(
    repo_id=REPO_ID, filename="resnet50_pytorch_model.bin"
)
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

model = resnet50(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, 90)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval().to(device)

In [ ]:
with open(CLASS_NAMES_PATH, encoding="utf-8") as f:
    class_names = json.load(f)
    print(len(class_names))

In [ ]:

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
def recognize_food(image_path, top_k=5):
    image = Image.open(image_path).convert("RGB")
    tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.inference_mode():
        probs = torch.softmax(model(tensor), dim=1)[0]

    scores, indices = torch.topk(probs, min(top_k, 90))
    return [
        {"label": class_names[i.item()], "confidence": round(s.item(), 4)}
        for s, i in zip(scores, indices)
    ]


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Find all images in DATA_FOLDER (including subfolders)
images = [
    path
    for path in DATA_FOLDER.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
]
print(images)

In [ ]:
print("go")
for image_path in images:
    print(f"\nImage: {image_path}")
    print("-" * 50)
    for result in recognize_food(image_path):
        print(f"{result['label']:<25} {result['confidence']:.2%}")